# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/georgy-com/Flyrank-Repo/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [7]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [14]:
import pandas as pd
import numpy as np

DATA_URL = (
    "https://huggingface.co/datasets/FlyRank/internship-starter/"
    "resolve/main/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Re-create the 'is_declining_label' column as it's not in the raw CSV
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}")

Dataset shape: (30000, 54)
Declining rate: 0.542


In [15]:
#stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["hand_rule_score"] = (
    stale * visible * df["impressions_90d"]
)

y = df["is_declining_label"].values

for k in (20, 50):
    score = precision_at_k(
        df["hand_rule_score"],
        y,
        k
    )
    print(f"Hand rule Precision@{k}: {score:.3f}")
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

Hand rule Precision@20: 0.900
Hand rule Precision@50: 0.680


In [16]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["hand_rule_score"] = (
    stale * visible * df["impressions_90d"]
)

y = df["is_declining_label"].values

for k in (20, 50):
    score = precision_at_k(
        df["hand_rule_score"],
        y,
        k
    )
    print(f"Hand rule Precision@{k}: {score:.3f}")

Hand rule Precision@20: 0.900
Hand rule Precision@50: 0.680


In [18]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [20]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [21]:
from sklearn.tree import DecisionTreeClassifier, export_text

depth_results = []

for depth in [2, 3, 4]:

    model = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X, y)

    scores = model.predict_proba(X)[:, 1]

    p20 = precision_at_k(scores, y, 20)
    p50 = precision_at_k(scores, y, 50)

    depth_results.append({
        "max_depth": depth,
        "Precision@20": p20,
        "Precision@50": p50
    })

depth_results_df = pd.DataFrame(depth_results)

print(depth_results_df.to_string(index=False))

 max_depth  Precision@20  Precision@50
         2          0.55          0.60
         3          0.70          0.72
         4          0.60          0.68


In [22]:
for depth in [3, 4]:

    model = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X, y)

    print("\n" + "=" * 70)
    print(f"DECISION TREE — MAX DEPTH {depth}")
    print("=" * 70)

    print(
        export_text(
            model,
            feature_names=features
        )
    )


DECISION TREE — MAX DEPTH 3
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0


DECISION TREE — MAX DEPTH 4
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50


In [23]:
# Remove impressions_90d
features_no_impressions = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

X_no_impressions = (
    df[features_no_impressions]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [24]:
tree_no_impressions = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

tree_no_impressions.fit(
    X_no_impressions,
    y
)

scores_no_impressions = (
    tree_no_impressions
    .predict_proba(X_no_impressions)[:, 1]
)

In [25]:
for k in (20, 50):

    score = precision_at_k(
        scores_no_impressions,
        y,
        k
    )

    print(
        f"Without impressions_90d "
        f"Precision@{k}: {score:.3f}"
    )

Without impressions_90d Precision@20: 0.850
Without impressions_90d Precision@50: 0.740


In [26]:
print(
    export_text(
        tree_no_impressions,
        feature_names=features_no_impressions
    )
)

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- class: 0
|   |   |--- word_count >  669.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- avg_position <= 38.45
|   |   |   |--- class: 0
|   |   |--- avg_position >  38.45
|   |   |   |--- class: 0



In [28]:
# Identify the First Split
first_split_index = tree_no_impressions.tree_.feature[0]

if first_split_index >= 0:
    first_split_feature = (
        features_no_impressions[first_split_index]
    )
    print(
        f"First split feature: {first_split_feature}"
    )
else:
    print("The tree did not make a split.")

First split feature: avg_position


In [29]:
# Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows: {len(X_test):,}")

print(
    f"Training declining rate: "
    f"{y_train.mean():.3f}"
)

print(
    f"Testing declining rate: "
    f"{y_test.mean():.3f}"
)

Training rows: 24,000
Testing rows: 6,000
Training declining rate: 0.542
Testing declining rate: 0.542


In [30]:
#Train the depth-2 model on training data
tree_train = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_train.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=2, random_state=42)

In [31]:
tree_train = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_train.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=2, random_state=42)

In [33]:
# Compare In-Sample vs Test Performance
train_scores = tree_train.predict_proba(X_train)[:, 1]
test_scores = tree_train.predict_proba(X_test)[:, 1]

comparison = []

for k in (20, 50):

    train_precision = precision_at_k(
        train_scores,
        y_train,
        k
    )

    test_precision = precision_at_k(
        test_scores,
        y_test,
        k
    )

    comparison.append({
        "K": k,
        "Train Precision": train_precision,
        "Test Precision": test_precision,
        "Gap": train_precision - test_precision
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df.to_string(index=False))

 K  Train Precision  Test Precision   Gap
20             0.50            0.65 -0.15
50             0.52            0.62 -0.10


In [35]:
# Compare Hand Rule vs Model on the Test Set
hand_rule_test = (
    (
        (df.loc[X_test.index, "days_since_last_update"] >= 180)
        .astype(int)
    )
    *
    (
        (df.loc[X_test.index, "impressions_90d"] >= 500)
        .astype(int)
    )
    *
    df.loc[X_test.index, "impressions_90d"]
)

In [37]:
hand_rule_test = (
    (
        (df.loc[X_test.index, "days_since_last_update"] >= 180)
        .astype(int)
    )
    *
    (
        (df.loc[X_test.index, "impressions_90d"] >= 500)
        .astype(int)
    )
    *
    df.loc[X_test.index, "impressions_90d"]
)

In [38]:
# Final Experiment Summary Table
summary = []

# Hand rule
for k in (20, 50):

    score = precision_at_k(
        df["hand_rule_score"],
        y,
        k
    )

    summary.append({
        "Model": "Hand rule",
        "Evaluation": "In-sample",
        "K": k,
        "Precision": score
    })


# Decision trees
for depth in [2, 3, 4]:

    model = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X, y)

    scores = model.predict_proba(X)[:, 1]

    for k in (20, 50):

        score = precision_at_k(
            scores,
            y,
            k
        )

        summary.append({
            "Model": f"Decision tree depth {depth}",
            "Evaluation": "In-sample",
            "K": k,
            "Precision": score
        })


summary_df = pd.DataFrame(summary)

summary_df["Precision"] = summary_df["Precision"].round(3)

print(summary_df.to_string(index=False))

                Model Evaluation  K  Precision
            Hand rule  In-sample 20       0.90
            Hand rule  In-sample 50       0.68
Decision tree depth 2  In-sample 20       0.55
Decision tree depth 2  In-sample 50       0.60
Decision tree depth 3  In-sample 20       0.70
Decision tree depth 3  In-sample 50       0.72
Decision tree depth 4  In-sample 20       0.60
Decision tree depth 4  In-sample 50       0.68


## My Experiment

I tested whether increasing the decision tree's maximum depth from 2 to 3 and 4 would improve Precision@50. I also tested whether removing `impressions_90d` and adding `engagement_rate` would change the feature used for the tree's first split. Finally, I compared the readable tree with the hand rule using a held-out 20% test set.

### What I observed

Increasing the tree depth changed the number of decision rules available to the model and resulted in a Precision@50 of **[INSERT YOUR DEPTH-3 RESULT]** at depth 3 and **[INSERT YOUR DEPTH-4 RESULT]** at depth 4, compared with **[INSERT YOUR DEPTH-2 RESULT]** for the depth-2 model.

After removing `impressions_90d`, the tree selected **[INSERT ACTUAL FIRST-SPLIT FEATURE]** as its first split. This shows that the model's readable decision structure can change substantially depending on which signals are available.

On the held-out test set, the depth-2 tree achieved a Precision@50 of **[INSERT TEST RESULT]**, compared with **[INSERT HAND-RULE TEST RESULT]** for the hand rule.

### Interpretation

The experiment suggests that allowing a deeper tree can provide additional decision rules, but greater complexity does not automatically mean a better ranking. The held-out test results are more useful for judging whether the observed pattern generalizes beyond the data used to fit the model.

The hand rule remains a useful baseline because it is simple and easy to explain. The decision tree provides a different advantage: it can discover combinations of observable features that may not be obvious when writing a rule manually.

These results are directional and specific to the starter dataset. They do not establish causal relationships or explain how Google's ranking algorithm works.
